# Tạo Beatmap Osu với BeatHeritage

Notebook này là một bản demo tương tác của mô hình tạo beatmap osu! do OliBomby phát triển và được SourceCode54 xây dựng lại.  
Mô hình này có thể tạo ra hit objects, hitsounds, timing, kiai times và SVs cho cả 4 chế độ chơi (gamemodes).  
Bạn có thể tải lên một beatmap để cung cấp thêm ngữ cảnh cho mô hình hoặc remap một phần của beatmap.  


### Hướng dẫn chạy:

* Đọc và chấp nhận các quy tắc khi sử dụng công cụ này  
* Đảm bảo sử dụng GPU runtime: __Nhấn Runtime >> Change Runtime Type >> Chọn GPU__  
* Nhấn ▶️ bên trái mỗi ô để thực thi lệnh  
* __Tải lên tệp âm thanh__: Chọn tệp .mp3 hoặc .ogg từ máy tính của bạn  
* __Tải lên beatmap__ (tùy chọn): Chọn tệp .osu từ máy tính của bạn  
* __Cấu hình__: Chọn các tham số tạo beatmap để kiểm soát phong cách beatmap đầu ra  
* Tạo beatmap bằng cách chạy ô __Generate Beatmap__ (quá trình có thể mất vài phút tùy vào độ dài bài hát)  


In [ ]:
#@title Setup Environment { display-mode: "form" }
#@markdown ### Hãy cận trọng khi sử dụng mô hình.
i_accept_the_rules = False # @param {type:"boolean"}

assert i_accept_the_rules, "Trước tiên, hãy đọc và chấp nhận các quy tắc!"

!git clone https://github.com/hongminh54/BeatHeritage.git
%cd BeatHeritage

!pip install -r requirements.txt

from google.colab import files

import os
import glob
from hydra import compose, initialize
from osuT5.osuT5.event import ContextType
from inference import main

output_path = "output"
input_audio = ""
input_beatmap = ""

In [ ]:
#@title Tải Lên Âm Thanh { display-mode: "form" }
#@markdown Đây là bài hát để tạo beatmap. Vui lòng tải lên tệp .mp3 hoặc .ogg.

def upload_audio():
    data = list(files.upload().keys())
    if len(data) > 1:
        print('Nhiều tệp đã được tải lên; chỉ sử dụng một tệp')
    file = data[0]
    if not file.endswith('.mp3') and not file.endswith('.ogg'):
        print('Định dạng tệp không hợp lệ. Vui lòng tải lên tệp .mp3 hoặc .ogg.')
        return ""
    return data[0]

input_audio = upload_audio()

In [ ]:
#@title (Tùy chọn) Tải Lên Beatmap { display-mode: "form" }
#@markdown Bước này là bắt buộc nếu bạn muốn sử dụng `in_context` hoặc `add_to_beatmap` để cung cấp thông tin bổ sung cho mô hình.
#@markdown Nó cũng sẽ tự động điền vào bất kỳ siêu dữ liệu bị thiếu và các giá trị chưa xác định trong cấu hình bằng thông tin từ beatmap tham chiếu.
#@markdown Vui lòng tải lên tệp .osu. Bạn có thể tìm thấy tệp .osu trong thư mục bài hát trên phiên bản stable hoặc bằng cách sử dụng File > Edit Externally trong lazer
use_reference_beatmap = False # @param {type:"boolean"}

def upload_beatmap():
    data = list(files.upload().keys())
    if len(data) > 1:
        print('Nhiều tệp đã được tải lên; chỉ sử dụng một tệp')
    file = data[0]
    if not file.endswith('.osu'):
        print('Định dạng tệp không hợp lệ. Vui lòng tải lên tệp .osu.\nTrong phiên bản stable, bạn có thể tìm thấy tệp .osu trong thư mục bài hát (File > Open Song Folder).\nTrong phiên bản lazer, bạn có thể tìm thấy tệp .osu bằng cách sử dụng File > Edit Externally.')
        return ""
    return file

if use_reference_beatmap:
    input_beatmap = upload_beatmap()
else:
    input_beatmap = ""

In [ ]:
#@title Chỉnh sửa và tạo map { display-mode: "form" }

#@markdown #### Bạn có thể nhập -1 để giữ giá trị không xác định.
#@markdown ---
#@markdown Chọn mode để tạo map.
gamemode = "standard" # @param ["standard", "taiko", "catch the beat", "mania"]
#@markdown Đây là mức độ khó (Star Rating) mong muốn của beatmap. Giá trị này có thể thay đổi tùy theo cường độ của bài hát và các cài đặt khác.
difficulty = 5 # @param {type:"number"}
#@markdown Đây là ID người dùng của mapper đã được xếp hạng mà bạn muốn mô phỏng phong cách mapping. Bạn có thể tìm thấy ID này trong URL của trang hồ sơ mapper.
mapper_id = -1 # @param {type:"integer"}
#@markdown Đây là năm mà bạn muốn beatmap được tạo theo. Giá trị phải nằm trong khoảng từ 2007 đến 2023.
year = 2023 # @param {type:"integer"}
#@markdown Đây là tùy chọn để xác định liệu beatmap có được thêm hiệu ứng âm thanh (hitsound) hay không. Tùy chọn này chỉ hoạt động cho chế độ Mania.
hitsounded = True # @param {type:"boolean"}
#@markdown Đây là hệ số nhân tốc độ thanh trượt (slider) toàn cục cho beatmap.
slider_multiplier = 1.7 # @param {type:"slider", min:0.4, max:3.6, step:0.1}
#@markdown Đây là kích thước vòng tròn (CS) của beatmap.
circle_size = 4 # @param {type:"number"}
#@markdown Đây là số lượng phím cho beatmap mania. Chỉ áp dụng cho chế độ mania.
keycount = 4 # @param {type:"slider", min:1, max:18, step:1}
#@markdown Đây là tỷ lệ giữa nốt giữ (hold notes) và nốt tròn (circles) trong beatmap. Giá trị nên nằm trong khoảng [0,1]. Chỉ áp dụng cho chế độ mania.
hold_note_ratio = -1 # @param {type:"number"}
#@markdown Đây là tỷ lệ giữa số lần thay đổi tốc độ cuộn và số lượng nốt trong beatmap. Giá trị nên nằm trong khoảng [0,1]. Chỉ áp dụng cho chế độ mania và taiko.
scroll_speed_ratio = -1 # @param {type:"number"}
#@markdown Đây là các mô tả của beatmap. Các mô tả này được sử dụng để diễn tả phong cách của beatmap. Bạn có thể tìm thấy tất cả các mô tả có sẵn [tại đây](https://omdb.nyahh.net/descriptors/).
descriptor_1 = '' # @param ["slider only", "circle only", "collab", "megacollab", "marathon", "gungathon", "multi-song", "variable timing", "accelerating bpm", "time signatures", "storyboard", "storyboard gimmick", "keysounds", "download unavailable", "custom skin", "featured artist", "custom song", "style", "messy", "geometric", "grid snap", "hexgrid", "freeform", "symmetrical", "old-style revival", "clean", "slidershapes", "distance snapped", "iNiS-style", "avant-garde", "perfect stacks", "ninja spinners", "simple", "chaotic", "repetition", "progression", "high contrast", "improvisation", "playfield usage", "playfield constraint", "video gimmick", "difficulty spike", "low sv", "high sv", "colorhax", "tech", "slider tech", "complex sv", "reading", "visually dense", "overlap reading", "alt", "jump aim", "sharp aim", "wide aim", "linear aim", "aim control", "flow aim", "precision", "finger control", "complex snap divisors", "bursts", "streams", "spaced streams", "cutstreams", "stamina", "mapping contest", "tournament custom", "tag", "port"] {allow-input: true}
descriptor_2 = '' # @param ["slider only", "circle only", "collab", "megacollab", "marathon", "gungathon", "multi-song", "variable timing", "accelerating bpm", "time signatures", "storyboard", "storyboard gimmick", "keysounds", "download unavailable", "custom skin", "featured artist", "custom song", "style", "messy", "geometric", "grid snap", "hexgrid", "freeform", "symmetrical", "old-style revival", "clean", "slidershapes", "distance snapped", "iNiS-style", "avant-garde", "perfect stacks", "ninja spinners", "simple", "chaotic", "repetition", "progression", "high contrast", "improvisation", "playfield usage", "playfield constraint", "video gimmick", "difficulty spike", "low sv", "high sv", "colorhax", "tech", "slider tech", "complex sv", "reading", "visually dense", "overlap reading", "alt", "jump aim", "sharp aim", "wide aim", "linear aim", "aim control", "flow aim", "precision", "finger control", "complex snap divisors", "bursts", "streams", "spaced streams", "cutstreams", "stamina", "mapping contest", "tournament custom", "tag", "port"] {allow-input: true}
descriptor_3 = '' # @param ["slider only", "circle only", "collab", "megacollab", "marathon", "gungathon", "multi-song", "variable timing", "accelerating bpm", "time signatures", "storyboard", "storyboard gimmick", "keysounds", "download unavailable", "custom skin", "featured artist", "custom song", "style", "messy", "geometric", "grid snap", "hexgrid", "freeform", "symmetrical", "old-style revival", "clean", "slidershapes", "distance snapped", "iNiS-style", "avant-garde", "perfect stacks", "ninja spinners", "simple", "chaotic", "repetition", "progression", "high contrast", "improvisation", "playfield usage", "playfield constraint", "video gimmick", "difficulty spike", "low sv", "high sv", "colorhax", "tech", "slider tech", "complex sv", "reading", "visually dense", "overlap reading", "alt", "jump aim", "sharp aim", "wide aim", "linear aim", "aim control", "flow aim", "precision", "finger control", "complex snap divisors", "bursts", "streams", "spaced streams", "cutstreams", "stamina", "mapping contest", "tournament custom", "tag", "port"] {allow-input: true}
#@markdown Đây là các mô tả tiêu cực của beatmap. Các mô tả tiêu cực được sử dụng để chỉ ra những yếu tố mà beatmap không nên có. Chúng chỉ hoạt động khi `cfg_scale` lớn hơn 1.
negative_descriptor_1 = '' # @param ["slider only", "circle only", "collab", "megacollab", "marathon", "gungathon", "multi-song", "variable timing", "accelerating bpm", "time signatures", "storyboard", "storyboard gimmick", "keysounds", "download unavailable", "custom skin", "featured artist", "custom song", "style", "messy", "geometric", "grid snap", "hexgrid", "freeform", "symmetrical", "old-style revival", "clean", "slidershapes", "distance snapped", "iNiS-style", "avant-garde", "perfect stacks", "ninja spinners", "simple", "chaotic", "repetition", "progression", "high contrast", "improvisation", "playfield usage", "playfield constraint", "video gimmick", "difficulty spike", "low sv", "high sv", "colorhax", "tech", "slider tech", "complex sv", "reading", "visually dense", "overlap reading", "alt", "jump aim", "sharp aim", "wide aim", "linear aim", "aim control", "flow aim", "precision", "finger control", "complex snap divisors", "bursts", "streams", "spaced streams", "cutstreams", "stamina", "mapping contest", "tournament custom", "tag", "port"] {allow-input: true}
negative_descriptor_2 = '' # @param ["slider only", "circle only", "collab", "megacollab", "marathon", "gungathon", "multi-song", "variable timing", "accelerating bpm", "time signatures", "storyboard", "storyboard gimmick", "keysounds", "download unavailable", "custom skin", "featured artist", "custom song", "style", "messy", "geometric", "grid snap", "hexgrid", "freeform", "symmetrical", "old-style revival", "clean", "slidershapes", "distance snapped", "iNiS-style", "avant-garde", "perfect stacks", "ninja spinners", "simple", "chaotic", "repetition", "progression", "high contrast", "improvisation", "playfield usage", "playfield constraint", "video gimmick", "difficulty spike", "low sv", "high sv", "colorhax", "tech", "slider tech", "complex sv", "reading", "visually dense", "overlap reading", "alt", "jump aim", "sharp aim", "wide aim", "linear aim", "aim control", "flow aim", "precision", "finger control", "complex snap divisors", "bursts", "streams", "spaced streams", "cutstreams", "stamina", "mapping contest", "tournament custom", "tag", "port"] {allow-input: true}
negative_descriptor_3 = '' # @param ["slider only", "circle only", "collab", "megacollab", "marathon", "gungathon", "multi-song", "variable timing", "accelerating bpm", "time signatures", "storyboard", "storyboard gimmick", "keysounds", "download unavailable", "custom skin", "featured artist", "custom song", "style", "messy", "geometric", "grid snap", "hexgrid", "freeform", "symmetrical", "old-style revival", "clean", "slidershapes", "distance snapped", "iNiS-style", "avant-garde", "perfect stacks", "ninja spinners", "simple", "chaotic", "repetition", "progression", "high contrast", "improvisation", "playfield usage", "playfield constraint", "video gimmick", "difficulty spike", "low sv", "high sv", "colorhax", "tech", "slider tech", "complex sv", "reading", "visually dense", "overlap reading", "alt", "jump aim", "sharp aim", "wide aim", "linear aim", "aim control", "flow aim", "precision", "finger control", "complex snap divisors", "bursts", "streams", "spaced streams", "cutstreams", "stamina", "mapping contest", "tournament custom", "tag", "port"] {allow-input: true}
#@markdown ---
#@markdown Nếu để True, beatmap được tạo sẽ được xuất dưới dạng tệp `.osz`. Ngược lại, nó sẽ được xuất dưới dạng tệp `.osu`.
export_osz = True # @param {type:"boolean"}
#@markdown Nếu để True, beatmap được tạo sẽ được thêm vào beatmap tham chiếu và beatmap tham chiếu sẽ được chỉnh sửa thay vì tạo một beatmap mới. Nó cũng sẽ tiếp tục bất kỳ hit object nào trước thời gian bắt đầu trong beatmap tham chiếu.
add_to_beatmap = False # @param {type:"boolean"}
#@markdown Đây là thời gian bắt đầu của beatmap tính bằng mili giây. Sử dụng thông số này để giới hạn việc tạo beatmap cho một phần cụ thể của bài hát.
start_time = -1 # @param {type:"integer"}
#@markdown Đây là thời gian kết thúc của beatmap tính bằng mili giây. Sử dụng thông số này để giới hạn việc tạo beatmap cho một phần cụ thể của bài hát.
end_time = -1 # @param {type:"integer"}
#@markdown Thông tin bổ sung cho mô hình:
#@markdown - TIMING: Cung cấp điểm timing cho mô hình. Điều này sẽ bỏ qua bước tạo điểm timing.
#@markdown - KIAI: Cung cấp thời gian kiai cho mô hình. Điều này sẽ bỏ qua bước tạo thời gian kiai.
#@markdown - MAP: Cung cấp hit objects cho mô hình. Điều này sẽ bỏ qua bước tạo hit objects.
#@markdown - GD: Cung cấp hit objects của một độ khó khác trong cùng một mapset cho mô hình (có thể thuộc chế độ chơi khác). Điều này sẽ cải thiện độ chính xác và tính nhất quán của nhịp điệu beatmap được tạo ra mà không sao chép beatmap tham chiếu.
#@markdown - NO_HS: Cung cấp hit objects không có hitsounds cho mô hình. Điều này sẽ sao chép các hit objects của beatmap tham chiếu và chỉ thêm hitsounds vào chúng.
in_context = "[NONE]" # @param ["[NONE]", "[TIMING]", "[TIMING,KIAI]", "[TIMING,KIAI,MAP]", "[GD,TIMING,KIAI]", "[NO_HS,TIMING,KIAI]"]
#@markdown Đây là loại đầu ra của beatmap. Bạn có thể chọn tạo mọi thứ hoặc chỉ tạo các điểm timing.
output_type = "[TIMING,KIAI,MAP]" # @param ["[TIMING,KIAI,MAP]", "[TIMING]"]
#@markdown Đây là thang đo của hướng dẫn không phân loại. Giá trị cao hơn sẽ làm cho mô hình có xu hướng tuân theo các mô tả và phong cách của mapper hơn.Giá trị `cfg_scale` cao hoặc một số tổ hợp cài đặt nhất định có thể tạo ra kết quả không mong muốn, vì vậy hãy sử dụng cẩn thận.
cfg_scale = 1 # @param {type:"slider", min:1, max:10, step:0.1}
#@markdown Đây là nhiệt độ của quá trình lấy mẫu. Nhiệt độ thấp hơn sẽ làm cho mô hình thận trọng hơn và tạo beatmap giống như một độ khó cao.Chỉ nên giảm giá trị này khi sử dụng `add_to_beatmap` và tạo các phần nhỏ của beatmap.
temperature = 1 # @param {type:"slider", min:0, max:1, step:0.01}
#@markdown Đây là seed ngẫu nhiên. Thay đổi giá trị này để tạo một beatmap khác với cùng cài đặt.
seed = -1 # @param {type:"integer"}
#@markdown ---
#@markdown Đây là số lượng beams cho thuật toán beam search trong bộ tạo timing. Giá trị cao hơn sẽ giúp timing chính xác hơn một chút nhưng sẽ làm chậm tốc độ xử lý.
timer_num_beams = 2 # @param {type:"slider", min:1, max:16, step:1}
#@markdown Đây là số lần lặp của bộ tạo timing. Giá trị cao hơn sẽ giúp timing chính xác hơn một chút nhưng sẽ làm chậm tốc độ xử lý.
timer_iterations = 20 # @param {type:"slider", min:1, max:100, step:1}
#@markdown Đây là ngưỡng độ chắc chắn yêu cầu cho các thay đổi BPM trong bộ tạo timing. Giá trị cao hơn sẽ làm giảm số lần thay đổi BPM.
timer_bpm_threshold = 0.1 # @param {type:"slider", min:0, max:1, step:0.1}
#@markdown ---

# Get actual parameters
a_gamemode = ["standard", "taiko", "catch the beat", "mania"].index(gamemode)
a_difficulty = None if difficulty == -1 else difficulty
a_mapper_id = None if mapper_id == -1 else mapper_id
a_year = None if year == -1 else year
a_circle_size = None if circle_size == -1 else circle_size
a_hold_note_ratio = None if hold_note_ratio == -1 else hold_note_ratio
a_scroll_speed_ratio = None if scroll_speed_ratio == -1 else scroll_speed_ratio
descriptors = [d for d in [descriptor_1, descriptor_2, descriptor_3] if d != '']
negative_descriptors = [d for d in [negative_descriptor_1, negative_descriptor_2, negative_descriptor_3] if d != '']

a_start_time = None if start_time == -1 else start_time
a_end_time = None if end_time == -1 else end_time
a_in_context = [ContextType(c.lower()) for c in in_context[1:-1].split(',')]
a_output_type = [ContextType(c.lower()) for c in output_type[1:-1].split(',')]
a_seed = None if seed == -1 else seed

# Validate stuff
if any(c in a_in_context for c in [ContextType.TIMING, ContextType.KIAI, ContextType.MAP, ContextType.SV, ContextType.GD, ContextType.NO_HS]) or add_to_beatmap:
    assert os.path.exists(input_beatmap), "Vui lòng tải lên một beatmap tham chiếu."
assert os.path.exists(input_audio), "Vui lòng tải lên một tệp âm thanh."

# Create config
with initialize(version_base="1.1", config_path="configs"):
    conf = compose(config_name="inference_v29")

# Do inference
conf.audio_path = input_audio
conf.output_path = output_path
conf.beatmap_path = input_beatmap
conf.gamemode = a_gamemode
conf.difficulty = a_difficulty
conf.mapper_id = a_mapper_id
conf.year = a_year
conf.hitsounded = hitsounded
conf.slider_multiplier = slider_multiplier
conf.circle_size = a_circle_size
conf.keycount = keycount
conf.hold_note_ratio = a_hold_note_ratio
conf.scroll_speed_ratio = a_scroll_speed_ratio
conf.descriptors = descriptors
conf.negative_descriptors = negative_descriptors
conf.export_osz = export_osz
conf.add_to_beatmap = add_to_beatmap
conf.start_time = a_start_time
conf.end_time = a_end_time
conf.in_context = a_in_context
conf.output_type = a_output_type
conf.cfg_scale = cfg_scale
conf.temperature = temperature
conf.seed = a_seed
conf.timer_num_beams = timer_num_beams
conf.timer_iterations = timer_iterations
conf.timer_bpm_threshold = timer_bpm_threshold

_, result_path, osz_path = main(conf)

if osz_path is not None:
    result_path = osz_path

if conf.add_to_beatmap:
    files.download(result_path)
else:
    files.download(result_path)
